#### 🥈 Pipeline de Transformação - Camada Silver (Prata)

##### Objetivo
Este notebook implementa o processo ETL da **Camada Bronze → Camada Silver (Prata)**, realizando:

* **Padronização** de nomes de grupos de lojas
* **Formatação** de tipos de dados (conversões seguras)
* **Enriquecimento** com colunas calculadas (mês, ano, colunas sem acentuação)
* **Deduplicação** de registros duplicados
* **Merge** incremental na tabela final `tb_prata`

##### Fluxo de Dados

tb_bronze → vw_group_store_silver → vw_silver → vw_silver_deduplicados → tb_clean_data


##### Catalog e Schema
* **Catalog**: `vendas_pecas`
* **Schema**: `camada_prata`

In [0]:
USE vendas_pecas.camada_prata;

#### 1️⃣ Padronização de Grupos de Lojas

##### Objetivo 
Corrigir inconsistências e variações nos nomes dos grupos de lojas.

##### Regras de Padronização
| Padrão Original | Nome Padronizado |
|------------------|------------------|
| *Tytan | Grupo Titan |
| *Atlântic | Grupo Atlantico |
| *Pampa | Grupo Pampas |
| *Serras | Grupo Serra |

##### Técnica
Usando `CASE WHEN` com `LIKE` para capturar variações de grafia.

##### Output
View temporária: `vw_group_store_silver` (adiciona coluna `new_grupo_loja`)

In [0]:
CREATE OR REPLACE TEMP VIEW vw_group_store_silver
AS 
SELECT *, 
CASE
  WHEN grupo_loja LIKE '%Tytan' THEN 'Grupo Titan'
  WHEN grupo_loja LIKE '%Atlântic' THEN 'Grupo Atlantico'
  WHEN grupo_loja LIKE '%Pampa' THEN 'Grupo Pampas'
  WHEN grupo_loja LIKE '%Serras' THEN 'Grupo Serra'
  ELSE grupo_loja
END AS new_grupo_loja
FROM 
  vendas_pecas.camada_bronze.tb_raw_data

#### 2️⃣ Formatação de Tipos de Dados e Enriquecimento

##### Objetivo
Converter tipos de dados para formatos apropriados e adicionar colunas calculadas.

##### Transformações Aplicadas

###### 🔄 Conversões de Tipo (usando `CAST`)
* `data_venda` → DATE
* `cliente_id` → BIGINT
* `valor_unitario` → DECIMAL(18,2)
* `quantidade`: tratamento de nulos `CASE WHEN`


###### ➕ Colunas Formatadas Adicionadas
* `mes_venda`: Extraído de `data_venda` usando `MONTH()`
* `produto_peca_sem_acento`: Remoção de acentuação usando `TRANSLATE()`

###### 🎯 Benefícios
* Permite análises temporais (mês/ano)
* Garante consistência de tipos de dados

##### Output
View temporária: `vw_silver` (dados prontos para carga)

In [0]:
CREATE OR REPLACE TEMP VIEW vw_silver
AS
WITH cte_silver AS
(
  SELECT 
    id_nf as IdNotaFiscal,
    CAST(data_venda as date) AS data_venda,
    CAST(cliente_id as bigint) AS cliente_id,
    cliente_nome,
    marca_carro,
    modelo_carro,
    produto_peca,
    CAST(valor_unitario as decimal(18,2)) AS valor_unitario,
    CAST(custo_unitario as decimal(18,2)) AS custo_unitario,
    coalesce (CAST(quantidade AS double), 0) AS quantidade,
    CAST(loja_id as bigint) AS loja_id,
    loja_nome,
    vendedor_id,
    vendedor_nome,
    new_grupo_loja AS grupo_loja,
    file_path,
    ingest_date
  FROM 
    vw_group_store_silver
)
SELECT
    IdNotaFiscal,
    data_venda,
    month(data_venda) AS mes_venda,
    year(data_venda) AS ano_venda,
    cliente_id,
    cliente_nome,
    marca_carro,
    modelo_carro,
    produto_peca,
    TRANSLATE(produto_peca, 'áàãâÁÀÃÂéèêÉÈÊíìîÍÌÎóòõôÓÒÕÔúùûÚÙÛçÇ', 'aaaaAAAAeeeEEEiiiIIIooooOOOOuuuUUUcC') AS produto_peca_sem_acento,
    valor_unitario,
    custo_unitario,
    quantidade,
    loja_id,
    loja_nome,
    grupo_loja,
    TRANSLATE(loja_nome, 'áàãâÁÀÃÂéèêÉÈÊíìîÍÌÎóòõôÓÒÕÔúùûÚÙÛçÇ', 'aaaaAAAAeeeEEEiiiIIIooooOOOOuuuUUUcC') AS loja_nome_sem_acento, 
    vendedor_id,
    vendedor_nome,   
    file_path,
    ingest_date
FROM 
  cte_silver

#### 3️⃣ Deduplicação de Registros

##### Objetivo
Remover registros duplicados da camada Bronze, mantendo apenas a versão mais recente de cada nota fiscal.

##### Estratégia
* `Chave de Deduplicação`: `id_nf` (ID da Nota Fiscal)
* `Critério de Seleção`: Registro com a maior `data_venda` e `ingest_date`
* `Técnica`: Window Function `ROW_NUMBER()` particionado por `id_nf`

##### Output
View temporária: `vw_silver_deduplicados`

In [0]:
CREATE OR REPLACE TEMP VIEW vw_silver_deduplicados
AS
SELECT *
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY IdNotaFiscal
            ORDER BY data_venda DESC, ingest_date DESC
        ) AS rn
    FROM vw_silver
) temp_tb
WHERE rn = 1

#### 4️⃣ Carga Incremental na Tabela Prata

##### Objetivo
Persistir os dados transformados na tabela final `tb_prata` de forma incremental.

##### Estratégia: MERGE (Upsert)
* `Matched`: Atualiza registros existentes (sobrescreve com dados mais recentes)
* `Not Matched`: Insere novos registros
* `Chave de Junção`: `IdNotaFiscal`

##### Recursos Utilizados
* `With Schema Evolution`: Permite evolução automática do schema da tabela
* `Update Set *`: Atualiza todas as colunas automaticamente
* `Insert *`: Insere todas as colunas automaticamente

##### Benefícios
* `Idempotência`: Pode ser executado múltiplas vezes sem duplicar dados
* `Performance`: Apenas processa registros novos ou modificados
* `Flexibilidade`: Schema pode evoluir sem quebrar o pipeline

##### Output
Tabela persistida: `vendas_pecas.camada_prata.tb_clean_data`

In [0]:
MERGE INTO tb_clean_data cd
USING vw_silver_deduplicados vs 
ON cd.IdNotaFiscal = vs.IdNotaFiscal
WHEN 
  MATCHED
THEN
  UPDATE SET *
WHEN
  NOT MATCHED
THEN 
  INSERT *

In [0]:
-- CREATE OR REPLACE TEMP VIEW vw_invalidos
-- AS
-- SELECT *
-- FROM vw_silver VW
-- WHERE VW.quantidade IS NULL or
-- VW.quantidade <= 0 or
-- VW.valor_unitario <= 0 or
-- VW.custo_unitario <= 0

In [0]:
-- CREATE OR REPLACE TEMP VIEW vw_validos
-- AS
-- SELECT *
-- FROM vw_silver VW
-- WHERE VW.quantidade IS NOT NULL and
-- VW.quantidade > 0 and
-- VW.valor_unitario > 0 and
-- VW.custo_unitario > 0

In [0]:
-- USE vendas_pecas.camada_prata;

-- CREATE OR REPLACE TABLE tb_quarentena
-- AS
-- SELECT *
-- FROM vw_invalidos